## Imports & setup

In [1]:
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import f1_score, roc_auc_score

SEED = 42
np.random.seed(SEED)

ID_COL = "ID"
TARGET = "is_climate_sensitive"
N_FOLDS = 5

## Load data

In [2]:
train = pd.read_csv(r"C:\Users\HomePC\Desktop\challenge\data\Train.csv")
test = pd.read_csv(r"C:\Users\HomePC\Desktop\challenge\data\Test.csv")
climate = pd.read_csv(r"C:\Users\HomePC\Desktop\challenge\data\climate_features.csv")

print(f"Train:   {train.shape}  \nTest:    {test.shape} \nClimate: {climate.shape}")
train.head()

Train:   (3146, 13)  
Test:    (1030, 12) 
Climate: (4176, 18)


,ID,zone,gender,deathdate,age,avg_temperature,max_temperature,min_temperature,precipitation,latitude,longitude,location,is_climate_sensitive
0,ID_1FE951BD,Rural,Male,2007-10-02,30.0,21.137037,25.569017,18.451642,0.951601,0.584646,33.546344,"Izimba, Iganga, Uganda",1
1,ID_5927A443,Rural,Male,2008-01-05,2.0,24.062790,29.312026,18.977212,0.053352,0.549565,33.490326,"Nawanzu, Iganga, Uganda",1
2,ID_C4E67025,Rural,Female,2008-02-14,31.0,22.407868,26.746058,18.112977,0.090349,0.573420,33.483655,"Buluza, Iganga, Uganda",1
3,ID_44DC5CE9,Rural,Female,2008-02-15,1.0,23.315198,28.056889,18.955708,0.286814,0.549565,33.490326,"Nawanzu, Iganga, Uganda",1
4,ID_33C677BB,Rural,Female,2008-02-15,69.0,23.315198,28.056889,18.955708,0.286814,1.072343,34.226647,"Magada, Mbale City, Bugisa sub-region, Eastern...",0


## Light cleaning + merge

In [3]:
# hot_days_30d is constant (always 0) across all rows -> zero signal, drop it
climate = climate.drop(columns=["hot_days_30d", "deathdate"])

train = train.merge(climate, on=ID_COL, how="left")
test = test.merge(climate, on=ID_COL, how="left")

for df in (train, test):
    df["deathdate"] = pd.to_datetime(df["deathdate"])

print("Missing values (train):", train.isnull().sum().sum())
print("Missing values (test):", test.isnull().sum().sum())

Missing values (train): 0
Missing values (test): 0


## Minimal feature engineering

In [4]:
def add_basic_features(df):
    df = df.copy()
    df["month"] = df["deathdate"].dt.month
    df["day_of_year"] = df["deathdate"].dt.dayofyear
    df["temperature_range"] = df["max_temperature"] - df["min_temperature"]
    df["is_infant"] = (df["age"] <= 1).astype(int)
    df["is_child"] = (df["age"] <= 5).astype(int)
    df["is_elderly"] = (df["age"] >= 65).astype(int)
    return df

train = add_basic_features(train)
test = add_basic_features(test)

## Define features

In [5]:
# `location` dropped as a direct feature: 39 train / 11 test locations,
# only 1 overlapping -> too close to a leak. lat/long keep spatial signal.
DROP_COLS = [ID_COL, TARGET, "deathdate", "location"]

categorical_features = ["zone", "gender"]
numeric_features = [c for c in train.columns
                     if c not in DROP_COLS + categorical_features]

X = train[categorical_features + numeric_features].copy()
y = train[TARGET].copy()
groups = train["location"].copy()

X_test = test[categorical_features + numeric_features].copy()

for c in categorical_features:
    X[c] = X[c].astype("category")
    X_test[c] = X_test[c].astype("category")

print("Categorical:", categorical_features)
print("Numeric:", numeric_features)

Categorical: ['zone', 'gender']
Numeric: ['age', 'avg_temperature', 'max_temperature', 'min_temperature', 'precipitation', 'latitude', 'longitude', 'elevation', 'max_daily_rain_30d', 'ndvi_30d', 'ndvi_90d', 'rain_days_30d', 'rain_sum_30d', 'rain_sum_7d', 'rain_sum_90d', 'slope', 'tavg_30d', 'tavg_7d', 'tavg_90d', 'temp_range_mean_30d', 'tmax_30d', 'tmin_30d', 'month', 'day_of_year', 'temperature_range', 'is_infant', 'is_child', 'is_elderly']


## official metric

In [6]:
def final_score(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    return 0.60 * f1 + 0.40 * auc, f1, auc

## CV harness function

In [7]:
def run_cv(splitter, X, y, groups=None, label=""):
    scores, f1s, aucs = [], [], []
    split_args = (X, y, groups) if groups is not None else (X, y)
    for fold, (tr_idx, va_idx) in enumerate(splitter.split(*split_args)):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        model = HistGradientBoostingClassifier(
            random_state=SEED,
            class_weight="balanced",
            categorical_features="from_dtype",
        )
        model.fit(X_tr, y_tr)
        val_prob = model.predict_proba(X_va)[:, 1]

        score, f1, auc = final_score(y_va, val_prob)
        scores.append(score); f1s.append(f1); aucs.append(auc)
        print(f"  [{label}] fold {fold}: F1={f1:.4f}  AUC={auc:.4f}  Score={score:.4f}")

    print(f"  [{label}] MEAN Score={np.mean(scores):.4f} "
          f"(F1={np.mean(f1s):.4f}, AUC={np.mean(aucs):.4f})  +/- {np.std(scores):.4f}\n")
    return scores

## Run both CV schemes (compare random vs grouped)

In [8]:
print("=== Plain random KFold (OPTIMISTIC) ===")
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
random_scores = run_cv(kf, X, y, label="random-KFold")

print("=== GroupKFold by location (REALISTIC) ===")
gkf = GroupKFold(n_splits=N_FOLDS)
group_scores = run_cv(gkf, X, y, groups=groups, label="group-KFold")

gap = np.mean(random_scores) - np.mean(group_scores)
print(f">>> Optimism gap (random - grouped): {gap:.4f}")
print(">>> TRUST THE GROUPED SCORE.")

=== Plain random KFold (OPTIMISTIC) ===
  [random-KFold] fold 0: F1=0.7679  AUC=0.7664  Score=0.7673
  [random-KFold] fold 1: F1=0.7926  AUC=0.7798  Score=0.7875
  [random-KFold] fold 2: F1=0.7859  AUC=0.7770  Score=0.7823
  [random-KFold] fold 3: F1=0.7911  AUC=0.7956  Score=0.7929
  [random-KFold] fold 4: F1=0.7921  AUC=0.7927  Score=0.7923
  [random-KFold] MEAN Score=0.7845 (F1=0.7859, AUC=0.7823)  +/- 0.0094

=== GroupKFold by location (REALISTIC) ===
  [group-KFold] fold 0: F1=0.7899  AUC=0.8038  Score=0.7955
  [group-KFold] fold 1: F1=0.7840  AUC=0.7855  Score=0.7846
  [group-KFold] fold 2: F1=0.7955  AUC=0.7952  Score=0.7954
  [group-KFold] fold 3: F1=0.7742  AUC=0.7853  Score=0.7787
  [group-KFold] fold 4: F1=0.7869  AUC=0.7645  Score=0.7780
  [group-KFold] MEAN Score=0.7864 (F1=0.7861, AUC=0.7869)  +/- 0.0077

>>> Optimism gap (random - grouped): -0.0020
>>> TRUST THE GROUPED SCORE.


## Quick threshold sweep

In [9]:
tr_idx, va_idx = next(gkf.split(X, y, groups))
X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

model = HistGradientBoostingClassifier(
    random_state=SEED, class_weight="balanced", categorical_features="from_dtype"
)
model.fit(X_tr, y_tr)
val_prob = model.predict_proba(X_va)[:, 1]

for t in [0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65]:
    _, f1, _ = final_score(y_va, val_prob, threshold=t)
    print(f"  threshold={t:.2f}  F1={f1:.4f}")

  threshold=0.35  F1=0.8112
  threshold=0.40  F1=0.8061
  threshold=0.45  F1=0.7922
  threshold=0.50  F1=0.7899
  threshold=0.55  F1=0.7752
  threshold=0.60  F1=0.7593
  threshold=0.65  F1=0.7466


## Train on full data, predict test

In [10]:
final_model = HistGradientBoostingClassifier(
    random_state=SEED, class_weight="balanced", categorical_features="from_dtype"
)
final_model.fit(X, y)

test_prob = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= 0.5).astype(int)

submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    "TargetF1": test_pred,
    "TargetRAUC": test_prob,
})
submission.to_csv("day1_submission.csv", index=False)

print("Saved day1_submission.csv")
print("Predicted positive rate on test:", test_pred.mean())
submission.head()

Saved day1_submission.csv
Predicted positive rate on test: 0.6213592233009708


,ID,TargetF1,TargetRAUC
0,ID_E760D84B,0,0.086798
1,ID_6EDEA907,1,0.826459
2,ID_B9FFC8D8,0,0.136449
3,ID_74C6C94E,0,0.307501
4,ID_0E02825D,1,0.861850


## Build features on training and test together 

In [11]:
import warnings; warnings.filterwarnings("ignore")

full = pd.concat([train.assign(_s="train"), test.assign(_s="test")], ignore_index=True)

# base features (Day 1)
full["month"] = full.deathdate.dt.month
full["day_of_year"] = full.deathdate.dt.dayofyear
full["temperature_range"] = full.max_temperature - full.min_temperature
full["is_infant"] = (full.age <= 1).astype(int)
full["is_child"] = (full.age <= 5).astype(int)
full["is_elderly"] = (full.age >= 65).astype(int)

# age features
full["age_group"] = pd.cut(full.age, [-1, 0, 1, 5, 15, 40, 65, 200], labels=False).astype(int)
full["log_age"] = np.log1p(full.age)
full["age_sq"] = full.age ** 2
full["under5_x_rain30"] = full.is_child * full.rain_sum_30d
full["under5_x_tavg30"] = full.is_child * full.tavg_30d
full["elderly_x_tmax30"] = full.is_elderly * full.tmax_30d
full["age_x_rain90"] = full.age * full.rain_sum_90d

# season features
full["month_sin"] = np.sin(2 * np.pi * full.month / 12)
full["month_cos"] = np.cos(2 * np.pi * full.month / 12)
full["doy_sin"] = np.sin(2 * np.pi * full.day_of_year / 365)
full["doy_cos"] = np.cos(2 * np.pi * full.day_of_year / 365)
full["rainy_season"] = full.month.isin([3, 4, 5, 9, 10, 11]).astype(int)

# climate dynamics
full["temp_trend_7_90"] = full.tavg_7d - full.tavg_90d
full["temp_trend_7_30"] = full.tavg_7d - full.tavg_30d
full["rain_ratio_7_30"] = full.rain_sum_7d / (full.rain_sum_30d + 1)
full["rain_ratio_30_90"] = full.rain_sum_30d / (full.rain_sum_90d + 1)
full["ndvi_change"] = full.ndvi_30d - full.ndvi_90d
full["heat_stress"] = full.tmax_30d - full.tmin_30d

DROP = ["ID", "is_climate_sensitive", "deathdate", "location", "_s"]
CATS = ["zone", "gender"]
feats = [c for c in full.columns if c not in DROP]

trn = full[full._s == "train"].reset_index(drop=True)
tst = full[full._s == "test"].reset_index(drop=True)

X = trn[feats].copy(); y = trn["is_climate_sensitive"].astype(int)
X_test = tst[feats].copy()
groups = trn["location"]
for c in CATS:
    X[c] = X[c].astype("category"); X_test[c] = X_test[c].astype("category")

print(len(feats), "features")

48 features


## regularized model + out of fold predictions

In [12]:
PARAMS = dict(random_state=SEED, class_weight="balanced", categorical_features="from_dtype",
              learning_rate=0.03, max_iter=300, max_depth=4,
              min_samples_leaf=40, l2_regularization=5.0)

folds = list(GroupKFold(n_splits=5).split(X, y, groups))
oof = np.zeros(len(X))
for a, b in folds:
    m = HistGradientBoostingClassifier(**PARAMS).fit(X.iloc[a], y.iloc[a])
    oof[b] = m.predict_proba(X.iloc[b])[:, 1]
print("OOF done")

OOF done


## threshold sweep on the official metric

In [13]:
for t in np.arange(0.15, 0.56, 0.05):
    f1s = [f1_score(y.iloc[b], (oof[b] >= t).astype(int)) for a, b in folds]
    aucs = [roc_auc_score(y.iloc[b], oof[b]) for a, b in folds]
    print(f"thr={t:.2f}  F1={np.mean(f1s):.4f}  Score={0.6*np.mean(f1s)+0.4*np.mean(aucs):.4f}  "
          f"pred_pos_rate={np.mean(oof >= t):.2f}")

thr=0.15  F1=0.8016  Score=0.8003  pred_pos_rate=0.94
thr=0.20  F1=0.8060  Score=0.8029  pred_pos_rate=0.89
thr=0.25  F1=0.8108  Score=0.8058  pred_pos_rate=0.83
thr=0.30  F1=0.8113  Score=0.8061  pred_pos_rate=0.76
thr=0.35  F1=0.8104  Score=0.8055  pred_pos_rate=0.71
thr=0.40  F1=0.8044  Score=0.8020  pred_pos_rate=0.66
thr=0.45  F1=0.7916  Score=0.7943  pred_pos_rate=0.62
thr=0.50  F1=0.7841  Score=0.7898  pred_pos_rate=0.58
thr=0.55  F1=0.7672  Score=0.7796  pred_pos_rate=0.54


## final model, predict test, save

In [14]:
final = HistGradientBoostingClassifier(**PARAMS).fit(X, y)
test_prob = final.predict_proba(X_test)[:, 1]

sub = pd.DataFrame({"ID": tst["ID"],
                    "TargetF1": (test_prob >= 0.5).astype(int),   # default threshold only
                    "TargetRAUC": test_prob})
sub.to_csv("day2_submission.csv", index=False)

In [15]:
feats_v3 = feats + ["location"]
X = trn[feats_v3].copy(); X_test = tst[feats_v3].copy()
for c in CATS + ["location"]:
    X[c] = X[c].astype("category")
    X_test[c] = pd.Categorical(tst[c], categories=X[c].cat.categories)

final = HistGradientBoostingClassifier(**PARAMS).fit(X, y)
test_prob = final.predict_proba(X_test)[:, 1]

sub = pd.DataFrame({"ID": tst["ID"],
                    "TargetF1": (test_prob >= 0.5).astype(int),
                    "TargetRAUC": test_prob})
sub.to_csv("day3_submission.csv", index=False)
sub.TargetF1.mean()

np.float64(0.6466019417475728)

In [16]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler

GB_PARAMS = dict(n_estimators=200, max_depth=4, learning_rate=0.03,
                 min_samples_leaf=30, subsample=0.8, random_state=SEED)

num = [c for c in feats if c not in CATS]
sc = StandardScaler(); ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

Xtr_num = sc.fit_transform(trn[num].fillna(0)); Xte_num = sc.transform(tst[num].fillna(0))
Xtr_cat = ohe.fit_transform(trn[CATS].astype(str)); Xte_cat = ohe.transform(tst[CATS].astype(str))
Xtr = np.hstack([Xtr_num, Xtr_cat]); Xte = np.hstack([Xte_num, Xte_cat])

final = GradientBoostingClassifier(**GB_PARAMS).fit(Xtr, y)
test_prob = final.predict_proba(Xte)[:, 1]

sub = pd.DataFrame({"ID": tst["ID"],
                    "TargetF1": (test_prob >= 0.5).astype(int),
                    "TargetRAUC": test_prob})
sub.to_csv("day4_submission.csv", index=False)
sub.TargetF1.mean()

np.float64(0.6912621359223301)